# 06. Capon versus Bartlett beamforming

The processing config exposes `beamforming_method`, defaulting to Capon. This notebook contrasts the two estimators that name selects: the conventional Bartlett beamformer and the adaptive Capon (minimum-variance) beamformer. On a synthetic pixel with two close scatterers it shows that Capon resolves them where Bartlett merges them, which is why the pipeline prefers it. The real estimator runs inside PyRat; the estimators here are the same mathematics on synthetic covariance.

**Modules exercised:** configuration.processing_config (TomogramConfiguration.beamforming_method), pipelines.processing_pipeline.tomogram (beamforming stage)

In [ ]:
import sys
from pathlib import Path

repo_root = Path("../..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy             as np
import matplotlib.pyplot as plt

try:
    import torch
    torch.manual_seed(0)
except ImportError:
    torch = None

RNG = np.random.default_rng(0)

%matplotlib inline
plt.rcParams.update({
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.grid"      : False,
    "image.cmap"     : "viridis",
})

print("repo root:", repo_root)


## Two closely spaced scatterers

We place two scatterers near each other in height so that resolution matters.

In [ ]:
n_tracks = 10
kz       = np.linspace(0.0, 0.50, n_tracks)
height_grid = np.linspace(-10.0, 60.0, 400)

true_heights = [22.0, 32.0]
n_looks      = 300

signal = np.zeros((n_tracks, n_looks), dtype=np.complex64)
for h in true_heights:
    a      = np.exp(1.0j * kz * h)[:, None]
    amp    = (RNG.standard_normal(n_looks) + 1.0j * RNG.standard_normal(n_looks)) / np.sqrt(2.0)
    signal += a * amp[None, :]
signal += 0.05 * (RNG.standard_normal(signal.shape) + 1.0j * RNG.standard_normal(signal.shape))
R = (signal @ signal.conj().T) / signal.shape[1]
print('covariance R:', R.shape)

## Both estimators

Bartlett: `a^H R a`. Capon: `1 / (a^H R^{-1} a)`. We add a small diagonal loading term to `R` before inversion for numerical stability, matching the defensive style of the codebase.

In [ ]:
steering = np.exp(1.0j * kz[:, None] * height_grid[None, :])

bartlett = np.einsum('th,ts,sh->h', steering.conj(), R, steering).real

loading  = 1e-3 * np.trace(R).real / n_tracks
R_inv    = np.linalg.inv(R + loading * np.eye(n_tracks))
capon    = 1.0 / np.einsum('th,ts,sh->h', steering.conj(), R_inv, steering).real

bartlett = bartlett / bartlett.max()
capon    = capon / capon.max()
print('diagonal loading:', float(loading))

## Resolution comparison

Capon should show two distinct peaks at the planted heights, while Bartlett tends to produce a single broad lobe.

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 3.8))
ax.plot(height_grid, bartlett, label='Bartlett')
ax.plot(height_grid, capon,    label='Capon')
for h in true_heights:
    ax.axvline(h, color='r', ls='--', lw=0.9)
ax.set_xlabel('height (m)'); ax.set_ylabel('normalised power')
ax.set_title('Capon vs Bartlett resolution')
ax.legend()
fig.tight_layout()
plt.show()

## Expected visual outcome

The Capon curve should display two separated peaks near 22 m and 32 m, whereas the Bartlett curve should be a single wide hump that fails to separate them. This illustrates why `beamforming_method` defaults to Capon for height resolution.